# HgFinance Risk Specialist QLoRA Training v1.2

This notebook trains the first Risk specialist adapter for HgFinance.

Qwen/Qwen2.5-14B-Instruct
↓
4-bit NF4 QLoRA
↓
Risk specialist data + Common behavioral replay
↓
hgfinance-risk-v1.2

This is NOT Common-only training and is NOT training on the production AWQ
checkpoint. The original Qwen/Qwen2.5-14B-Instruct lineage is loaded in
4-bit NF4 for QLoRA. The resulting adapter is intended for later
Qwen2.5-14B-Instruct-AWQ + LoRA serving and evaluation.

Risk authority contract:
- Hermes Risk supervisor is advisory.
- compliance-policy-worker interprets point-in-time Mandate, Restricted List,
  and Policy Store evidence.
- risk-runner performs deterministic market, liquidity, pre-trade,
  counterparty, and numeric checks.
- The deterministic Risk Engine owns binding APPROVE / RESIZE / REJECT.
- The Risk LLM must preserve the binding engine verdict and must not invent,
  upgrade, downgrade, or overwrite it.
- A frozen mandate snapshot is point-in-time advisory evidence only.
- Missing snapshot limits must not be replaced with defaults.
- Plain execution requests route to Trading/OMS; risk-validation questions
  belong to Risk. Order-time deterministic Risk Engine checks remain mandatory.

Production serving remains separate: vLLM max_model_len=8192. This notebook's
training MAX_LENGTH is an independent training sequence-length setting.

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile
import shutil

# 1) Drive mount
drive.mount("/content/drive")

# 2) 네가 방금 올린 ZIP
DRIVE_ZIP = Path("/content/drive/MyDrive/hgfinance_colab_training.zip")

if not DRIVE_ZIP.is_file():
    raise FileNotFoundError(f"ZIP not found: {DRIVE_ZIP}")

# 3) Colab runtime repo 생성
REPO_ROOT = Path("/content/multi_agent")

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

REPO_ROOT.mkdir(parents=True, exist_ok=True)

# 4) 압축 해제
with zipfile.ZipFile(DRIVE_ZIP) as z:
    z.extractall(REPO_ROOT)

# 5) 필수 파일 확인
checks = {
    "training": REPO_ROOT / "training",
    "qlora_script": REPO_ROOT / "scripts/qlora/train_specialist_qlora.py",
    "benchmarks": REPO_ROOT / "benchmarks/quantization",
}

print("REPO_ROOT:", REPO_ROOT)
for name, path in checks.items():
    print(f"{name:12s}: {path.exists()}")

assert all(path.exists() for path in checks.values())

print("\n✅ Colab training payload ready")

# Risk v1.2 dataset package extraction happens after the base payload extraction above.
from pathlib import Path
from zipfile import ZipFile

RISK_ZIP_PATH = Path("/content/drive/MyDrive/hgfinance_risk_training_v1_2.zip")
RISK_REPO_ROOT = Path("/content/multi_agent")
if not RISK_ZIP_PATH.is_file():
    raise FileNotFoundError(f"Risk dataset ZIP does not exist: {RISK_ZIP_PATH}")
RISK_REPO_ROOT.mkdir(parents=True, exist_ok=True)
with ZipFile(RISK_ZIP_PATH) as risk_archive:
    risk_archive.extractall(RISK_REPO_ROOT)
if not (RISK_REPO_ROOT / "hgfinance_risk_training_v1_2").is_dir():
    raise FileNotFoundError(
        "Risk dataset package did not extract to "
        f"{RISK_REPO_ROOT / 'hgfinance_risk_training_v1_2'}"
    )


In [ ]:
# 1. Runtime and GPU check
import os
import json
import math
import hashlib
import inspect
import shutil
import subprocess
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("A Colab CUDA GPU is required for training.")

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16_SUPPORTED = torch.cuda.is_bf16_supported()

print("GPU:", GPU_NAME)
print(f"VRAM: {VRAM_GIB:.2f} GiB")
print("BF16 supported:", BF16_SUPPORTED)

In [ ]:
# 2. Repository path setup
from pathlib import Path
import os

REPO_ROOT = Path(
    os.environ.get("HGFINANCE_REPO_ROOT", "/content/multi_agent")
).resolve()

if not REPO_ROOT.exists():
    raise FileNotFoundError(
        f"Repository root does not exist: {REPO_ROOT}"
    )

os.chdir(REPO_ROOT)
print("Repository:", REPO_ROOT)

In [ ]:
# 3. Install dependencies and configure Risk specialist training
%pip install -q -r training/qlora/requirements-colab.txt

import importlib.metadata as importlib_metadata

for package_name in ("torch", "transformers", "peft", "accelerate", "bitsandbytes", "datasets"):
    try:
        print(f"{package_name}: {importlib_metadata.version(package_name)}")
    except importlib_metadata.PackageNotFoundError:
        print(f"{package_name}: NOT INSTALLED")

import random
import numpy as np

BASE_MODEL = "Qwen/Qwen2.5-14B-Instruct"
ADAPTER_NAME = "hgfinance-risk-v1.2"
ADAPTER_VERSION = "v1.2"
TRAINING_MODE = "specialist"

RISK_DATA_DIR = REPO_ROOT / "hgfinance_risk_training_v1_2"
PREPARED_DIR = RISK_DATA_DIR / "prepared"
PREPARED_TRAIN_PATH = PREPARED_DIR / "train.jsonl"
PREPARED_VALIDATION_PATH = PREPARED_DIR / "validation.jsonl"
RISK_SOUL_PATH = RISK_DATA_DIR / "RISK_SOUL.md"
ROUTING_CONTRACT_PATH = RISK_DATA_DIR / "routing_contract.json"
VALIDATION_REPORT_PATH = RISK_DATA_DIR / "validation_report.json"
RISK_VALIDATOR_PATH = RISK_DATA_DIR / "validate_risk_dataset.py"
OUTPUT_DIR = REPO_ROOT / "training_runs" / "hgfinance-risk-v1.2"
BENCHMARK_ROOT = REPO_ROOT / "benchmarks" / "quantization"

EXPECTED_TRAIN_COUNT = 1350
EXPECTED_VALIDATION_COUNT = 150
MAX_LENGTH = 3072
NUM_TRAIN_EPOCHS = 1.0
LEARNING_RATE = 2e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
SEED = 66
OPTIMIZER = "paged_adamw_8bit"

QLORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
assert QLORA_R <= 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Repository:", REPO_ROOT)
print("Training mode:", TRAINING_MODE)
print("Adapter:", ADAPTER_NAME)
print("Risk train:", PREPARED_TRAIN_PATH)
print("Risk validation:", PREPARED_VALIDATION_PATH)

## Risk v1.2 dataset and sequence-length decision

This notebook uses the prepared Risk v1.2 package: 1,100 Risk specialist
training examples plus 250 Common behavioral replay examples, with 150
Risk-only validation examples. The prepared split is used unchanged.

MAX_LENGTH=3072 is the initial Risk v1.2 training limit. The tokenizer-first
audit is rerun over all 1,350 training and 150 validation examples; the Risk
dataset is not assumed to have the Common dataset's length distribution.
Any future over-limit record fails closed before model loading. No truncation,
record dropping, or assistant-answer shortening is allowed.

This training setting is separate from production vLLM max_model_len=8192.

In [ ]:
# 4. Risk package and prepared JSONL integrity
import hashlib
import json
import subprocess
import sys
from training.specialist.schema import DatasetValidationError, load_jsonl

REQUIRED_RISK_FILES = [
    PREPARED_TRAIN_PATH,
    PREPARED_VALIDATION_PATH,
    RISK_SOUL_PATH,
    ROUTING_CONTRACT_PATH,
    VALIDATION_REPORT_PATH,
    RISK_VALIDATOR_PATH,
    RISK_DATA_DIR / "metadata.json",
]
for required_path in REQUIRED_RISK_FILES:
    if not required_path.is_file():
        raise FileNotFoundError(f"Required Risk file is missing: {required_path}")

if not RISK_SOUL_PATH.read_text(encoding="utf-8").strip():
    raise DatasetValidationError("RISK_SOUL.md is empty")
routing_contract = json.loads(ROUTING_CONTRACT_PATH.read_text(encoding="utf-8"))
if not isinstance(routing_contract, dict):
    raise DatasetValidationError("routing_contract.json must contain an object")

RISK_VALIDATION_REPORT = json.loads(
    VALIDATION_REPORT_PATH.read_text(encoding="utf-8")
)
if RISK_VALIDATION_REPORT.get("status") != "PASS":
    raise DatasetValidationError(
        f"Risk validation report is not PASS: {RISK_VALIDATION_REPORT}"
    )
for report_key in (
    "duplicate_ids",
    "duplicate_sample_hashes",
    "normalized_user_overlap_train_validation",
):
    if int(RISK_VALIDATION_REPORT.get(report_key, -1)) != 0:
        raise DatasetValidationError(
            f"Risk validation report failed {report_key}: "
            f"{RISK_VALIDATION_REPORT.get(report_key)}"
        )

validator_result = subprocess.run(
    [sys.executable, str(RISK_VALIDATOR_PATH)],
    cwd=str(RISK_DATA_DIR),
    capture_output=True,
    text=True,
)
if validator_result.stdout:
    print(validator_result.stdout)
if validator_result.stderr:
    print(validator_result.stderr)
if validator_result.returncode != 0:
    raise DatasetValidationError(
        "validate_risk_dataset.py failed with exit code "
        f"{validator_result.returncode}"
    )

risk_train_examples = load_jsonl(
    PREPARED_TRAIN_PATH,
    source_dataset="risk_specialist",
)
risk_validation_examples = load_jsonl(
    PREPARED_VALIDATION_PATH,
    source_dataset="risk_specialist",
)
assert len(risk_train_examples) == EXPECTED_TRAIN_COUNT
assert len(risk_validation_examples) == EXPECTED_VALIDATION_COUNT

print("Risk validation report:", RISK_VALIDATION_REPORT["status"])
print("Risk train records:", len(risk_train_examples))
print("Risk validation records:", len(risk_validation_examples))
for split_name, examples in (
    ("train", risk_train_examples),
    ("validation", risk_validation_examples),
):
    print(
        split_name,
        "sample:",
        {
            "id": examples[0].record["id"],
            "category": examples[0].record.get("category"),
            "roles": [message["role"] for message in examples[0].messages],
        },
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
# 5. Held-out benchmark contamination checks
from training.specialist.contamination import check_contamination, require_clean

train_contamination = check_contamination(
    risk_train_examples,
    BENCHMARK_ROOT,
)
validation_contamination = check_contamination(
    risk_validation_examples,
    BENCHMARK_ROOT,
)
require_clean(train_contamination)
require_clean(validation_contamination)

BENCHMARK_CONTAMINATION = {
    "status": "PASS",
    "held_out": [
        "External-50",
        "Internal-v1",
        "Internal-v2 / EmployeeReasoning",
    ],
    "train": train_contamination,
    "validation": validation_contamination,
}
print("benchmark texts:", train_contamination["benchmark_text_count"])
print("train exact:", train_contamination["exact_count"])
print("train near:", train_contamination["near_count"])
print("validation exact:", validation_contamination["exact_count"])
print("validation near:", validation_contamination["near_count"])

In [ ]:
# 6. Resolve and pin the Hugging Face base revision before model loading
import os
from huggingface_hub import model_info

REQUESTED_BASE_REVISION = os.environ.get("HF_BASE_REVISION") or None
MODEL_INFO = model_info(BASE_MODEL, revision=REQUESTED_BASE_REVISION)
RESOLVED_BASE_REVISION = MODEL_INFO.sha
if not RESOLVED_BASE_REVISION:
    raise RuntimeError("Could not resolve a base model revision")
print("Base model:", BASE_MODEL)
print("Requested revision:", REQUESTED_BASE_REVISION)
print("Resolved revision:", RESOLVED_BASE_REVISION)

In [ ]:
# 7. Tokenizer-first full-sequence Risk audit; no model load before PASS
import math
from transformers import AutoTokenizer
from training.specialist.schema import DatasetValidationError

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    revision=RESOLVED_BASE_REVISION,
)
if tokenizer.chat_template is None:
    raise RuntimeError("Qwen tokenizer chat_template is required")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def full_chat_token_ids(messages):
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    encoded = tokenizer(
        rendered,
        add_special_tokens=False,
        truncation=False,
    )
    return list(encoded["input_ids"])

def percentile(values, fraction):
    ordered = sorted(values)
    index = max(
        0,
        min(len(ordered) - 1, math.ceil(len(ordered) * fraction) - 1),
    )
    return ordered[index]

all_risk_examples = [
    ("train", example) for example in risk_train_examples
] + [
    ("validation", example) for example in risk_validation_examples
]
lengths = []
over_limit = []
for split_name, example in all_risk_examples:
    token_length = len(full_chat_token_ids(example.messages))
    lengths.append(token_length)
    if token_length > MAX_LENGTH:
        over_limit.append(
            {
                "split": split_name,
                "sample_id": example.record["id"],
                "full_token_length": token_length,
            }
        )

TOKEN_LENGTH_AUDIT = {
    "example_count": len(lengths),
    "p50_token_length": percentile(lengths, 0.50),
    "p95_token_length": percentile(lengths, 0.95),
    "p99_token_length": percentile(lengths, 0.99),
    "max_token_length": max(lengths),
    "max_seq_length": MAX_LENGTH,
    "over_limit_count": len(over_limit),
    "over_limit_examples": over_limit,
}
print(json.dumps(TOKEN_LENGTH_AUDIT, indent=2))
if over_limit:
    for item in over_limit:
        print(
            "OVER_LIMIT",
            "split=",
            item["split"],
            "sample_id=",
            item["sample_id"],
            "full_token_length=",
            item["full_token_length"],
        )
    first = over_limit[0]
    raise DatasetValidationError(
        "Full sequence exceeds MAX_LENGTH; refusing model load. "
        f"sample_id={first['sample_id']} "
        f"full_token_length={first['full_token_length']} "
        f"max_seq_length={MAX_LENGTH}"
    )

In [ ]:
# 8. Load original Qwen base in 4-bit NF4 only after the audit passes
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

COMPUTE_DTYPE = (
    torch.bfloat16 if BF16_SUPPORTED else torch.float16
)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    revision=RESOLVED_BASE_REVISION,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
    trust_remote_code=True,
)
print("Loaded:", BASE_MODEL)
print("Revision:", RESOLVED_BASE_REVISION)
print(
    "Memory footprint:",
    f"{model.get_memory_footprint() / 1024**3:.2f} GiB",
)

In [ ]:
# 9. Explicit gradient checkpointing and PEFT configuration
import inspect
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model.config.use_cache = False
prepare_parameters = inspect.signature(
    prepare_model_for_kbit_training
).parameters
prepare_kwargs = {}
if "use_gradient_checkpointing" in prepare_parameters:
    prepare_kwargs["use_gradient_checkpointing"] = True
if "gradient_checkpointing_kwargs" in prepare_parameters:
    prepare_kwargs["gradient_checkpointing_kwargs"] = {
        "use_reentrant": False
    }
model = prepare_model_for_kbit_training(model, **prepare_kwargs)
if "use_gradient_checkpointing" not in prepare_parameters:
    enable_parameters = inspect.signature(
        model.gradient_checkpointing_enable
    ).parameters
    if "gradient_checkpointing_kwargs" in enable_parameters:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    else:
        model.gradient_checkpointing_enable()

peft_config = LoraConfig(
    r=QLORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
# Keep the external adapter identity in ADAPTER_NAME. Leave PEFT's internal
# adapter key at its default so save_pretrained(OUTPUT_DIR) writes root-level
# adapter artifacts consistently across PEFT versions.
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 10. Build prepared datasets and enforce assistant-only loss
from datasets import Dataset
from training.specialist.schema import DatasetValidationError

train_dataset = Dataset.from_list(
    [example.record for example in risk_train_examples]
)
validation_dataset = Dataset.from_list(
    [example.record for example in risk_validation_examples]
)

class AssistantOnlyCollator:
    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, features):
        rows = []
        for feature in features:
            messages = feature["messages"]
            full_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            prefix_text = tokenizer.apply_chat_template(
                messages[:-1],
                tokenize=False,
                add_generation_prompt=True,
            )
            sample_id = feature.get("id", "unknown")
            full = self.tokenizer(
                full_text,
                add_special_tokens=False,
                truncation=False,
            )
            full_length = len(full["input_ids"])
            if full_length > self.max_length:
                raise DatasetValidationError(
                    "Assistant completion cannot be safely truncated: "
                    f"sample_id={sample_id} "
                    f"full_token_length={full_length} "
                    f"max_seq_length={self.max_length}"
                )
            prefix = self.tokenizer(
                prefix_text,
                add_special_tokens=False,
                truncation=False,
            )
            if full["input_ids"][:len(prefix["input_ids"])] != prefix["input_ids"]:
                raise DatasetValidationError(
                    "Qwen chat template prefix is not a full-sequence prefix"
                )
            if full_length <= len(prefix["input_ids"]):
                raise DatasetValidationError(
                    f"Assistant completion is empty: sample_id={sample_id}"
                )
            labels = (
                [-100] * len(prefix["input_ids"])
                + full["input_ids"][len(prefix["input_ids"]):]
            )
            rows.append(
                {
                    "input_ids": full["input_ids"],
                    "attention_mask": full["attention_mask"],
                    "labels": labels,
                }
            )

        batch = self.tokenizer.pad(
            [
                {
                    "input_ids": row["input_ids"],
                    "attention_mask": row["attention_mask"],
                }
                for row in rows
            ],
            return_tensors="pt",
        )
        max_batch_length = batch["input_ids"].shape[1]
        labels = [
            row["labels"] + [-100] * (max_batch_length - len(row["labels"]))
            for row in rows
        ]
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

data_collator = AssistantOnlyCollator(tokenizer, MAX_LENGTH)
print(train_dataset)
print(validation_dataset)

In [ ]:
# 11. Trainer configuration
import inspect
from transformers import Trainer, TrainingArguments

training_parameters = inspect.signature(TrainingArguments).parameters
required_parameters = {
    "optim",
    "gradient_checkpointing",
    "gradient_checkpointing_kwargs",
}
missing_parameters = required_parameters - set(training_parameters)
if missing_parameters:
    raise RuntimeError(
        f"Transformers API lacks required controls: {sorted(missing_parameters)}"
    )

trainer_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / ".trainer"),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    optim=OPTIMIZER,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=BF16_SUPPORTED,
    fp16=not BF16_SUPPORTED,
    eval_strategy="epoch",
    save_strategy="no",
    report_to=[],
    seed=SEED,
    remove_unused_columns=False,
)
trainer = Trainer(
    model=model,
    args=trainer_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
)
print("Trainer configured")

In [ ]:
# 12. Final preflight — next cell is the only training cell
EFFECTIVE_BATCH_SIZE = (
    PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
)
EXPECTED_OPTIMIZER_STEPS = math.ceil(
    len(train_dataset) / GRADIENT_ACCUMULATION_STEPS
)
print("Training mode:", TRAINING_MODE)
print("Adapter:", ADAPTER_NAME)
print("Department: risk")
print("Train:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Base:", BASE_MODEL)
print("Revision:", RESOLVED_BASE_REVISION)
print("Quantization: 4-bit NF4, double quantization")
print(f"LoRA: r={QLORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print("Targets:", TARGET_MODULES)
print("Optimizer:", OPTIMIZER)
print("Gradient checkpointing: True")
print("MAX_LENGTH:", MAX_LENGTH)
print("Epochs:", NUM_TRAIN_EPOCHS)
print("Effective batch:", EFFECTIVE_BATCH_SIZE)
print("Expected optimizer steps (approx):", EXPECTED_OPTIMIZER_STEPS)

assert NUM_TRAIN_EPOCHS == 1.0
assert TRAINING_MODE == "specialist"
assert ADAPTER_NAME == "hgfinance-risk-v1.2"
assert len(train_dataset) == EXPECTED_TRAIN_COUNT
assert len(validation_dataset) == EXPECTED_VALIDATION_COUNT
assert OPTIMIZER == "paged_adamw_8bit"
assert model.config.use_cache is False

In [ ]:
# 13. TRAIN — intentionally separate and do not run during notebook preparation
train_result = trainer.train()
print(train_result)

In [ ]:
# 14. Evaluate Risk specialist training
eval_metrics = trainer.evaluate()
print(json.dumps(eval_metrics, indent=2, default=str))

In [ ]:
# 15. Save root-level adapter artifacts and Risk metadata
import importlib.metadata as importlib_metadata
import subprocess

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_DIR / "tokenizer")

def git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=REPO_ROOT,
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return "UNKNOWN"

def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "UNKNOWN"

prepared_train_sha256 = sha256_file(PREPARED_TRAIN_PATH)
prepared_validation_sha256 = sha256_file(PREPARED_VALIDATION_PATH)
metadata = {
    "project": "HgFinance",
    "training_mode": "specialist",
    "department": "risk",
    "dataset_version": "hgfinance-risk-training-v1.2",
    "adapter_name": ADAPTER_NAME,
    "adapter_version": ADAPTER_VERSION,
    "base_model": BASE_MODEL,
    "requested_base_revision": REQUESTED_BASE_REVISION,
    "resolved_base_revision": RESOLVED_BASE_REVISION,
    "prepared_train_sha256": prepared_train_sha256,
    "prepared_validation_sha256": prepared_validation_sha256,
    "risk_validation_report": RISK_VALIDATION_REPORT,
    "train_count": len(train_dataset),
    "validation_count": len(validation_dataset),
    "benchmark_contamination": BENCHMARK_CONTAMINATION,
    "held_out_benchmarks": [
        "External-50",
        "Internal-v1",
        "Internal-v2 / EmployeeReasoning",
    ],
    "risk_architecture": {
        "risk_supervisor_authority": "advisory",
        "binding_risk_authority": "deterministic Risk Engine",
        "frozen_mandate_snapshot": "point-in-time advisory evidence",
        "plain_execution_conversation_route": "Trading/OMS",
    },
    "lora": {
        "r": QLORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
        "max_production_rank": 32,
    },
    "quantization": {
        "load_in_4bit": True,
        "quant_type": "nf4",
        "double_quant": True,
        "compute_dtype": str(COMPUTE_DTYPE),
    },
    "optimizer": OPTIMIZER,
    "gradient_checkpointing": True,
    "gradient_checkpointing_use_reentrant": False,
    "use_cache": False,
    "eval_strategy": "epoch",
    "save_strategy": "no",
    "max_length": MAX_LENGTH,
    "epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "token_length_audit": TOKEN_LENGTH_AUDIT,
    "runtime": {
        "gpu": GPU_NAME,
        "vram_gib": VRAM_GIB,
        "torch": package_version("torch"),
        "transformers": package_version("transformers"),
        "peft": package_version("peft"),
        "accelerate": package_version("accelerate"),
        "bitsandbytes": package_version("bitsandbytes"),
        "datasets": package_version("datasets"),
    },
    "git_commit": git_commit(),
    "final_eval_metrics": eval_metrics,
}
(OUTPUT_DIR / "training_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False, default=str) + "\n",
    encoding="utf-8",
)
print(json.dumps({
    "adapter": ADAPTER_NAME,
    "adapter_model": str(OUTPUT_DIR / "adapter_model.safetensors"),
    "metadata": str(OUTPUT_DIR / "training_metadata.json"),
}, indent=2))

In [ ]:
# 16. Verify root-level adapter-only artifacts
required_artifacts = [
    OUTPUT_DIR / "adapter_model.safetensors",
    OUTPUT_DIR / "adapter_config.json",
    OUTPUT_DIR / "training_metadata.json",
]
for artifact in required_artifacts:
    if not artifact.is_file():
        raise FileNotFoundError(f"Missing adapter artifact: {artifact}")
for forbidden in ("pytorch_model.bin", "model.safetensors"):
    if (OUTPUT_DIR / forbidden).exists():
        raise RuntimeError(f"Full base-model artifact is forbidden: {forbidden}")
saved_metadata = json.loads(
    (OUTPUT_DIR / "training_metadata.json").read_text(encoding="utf-8")
)
assert saved_metadata["training_mode"] == "specialist"
assert saved_metadata["department"] == "risk"
assert saved_metadata["adapter_name"] == "hgfinance-risk-v1.2"
assert saved_metadata["train_count"] == EXPECTED_TRAIN_COUNT
assert saved_metadata["validation_count"] == EXPECTED_VALIDATION_COUNT
print("Root-level adapter-only artifact verification PASS")

In [ ]:
# 17. Risk-relevant sanity generation; not a formal benchmark
risk_validation_records = [
    example.record for example in risk_validation_examples
]

def _user_text(record):
    return "\n".join(
        message["content"]
        for message in record["messages"]
        if message["role"] == "user"
    ).casefold()

def _select_first_sanity_case(label, predicate):
    for record in risk_validation_records:
        if predicate(record):
            return record
    raise RuntimeError(
        "Could not find deterministic Risk sanity case for " + label
    )

selected_sanity_cases = [
    (
        "engine_verdict",
        _select_first_sanity_case(
            "engine_verdict",
            lambda record: (
                record.get("subcategory")
                == "risk_engine_result_interpretation"
                and str(record.get("engine_verdict", "")).upper()
                in {"APPROVE", "RESIZE", "REJECT"}
            ),
        ),
    ),
    (
        "frozen_mandate_snapshot",
        _select_first_sanity_case(
            "frozen_mandate_snapshot",
            lambda record: (
                record.get("subcategory")
                == "frozen_mandate_snapshot"
            ),
        ),
    ),
    (
        "sell_risk_validation",
        _select_first_sanity_case(
            "sell_risk_validation",
            lambda record: (
                record.get("subcategory")
                == "buy_sell_risk_validation"
                and "매도" in _user_text(record)
            ),
        ),
    ),
    (
        "plain_execution_routing",
        _select_first_sanity_case(
            "plain_execution_routing",
            lambda record: (
                record.get("subcategory")
                == "routing_authority_boundary"
                and str(record.get("routing", "")).upper() == "TRADING"
            ),
        ),
    ),
]
selected_sanity_ids = [record["id"] for _, record in selected_sanity_cases]
assert len(selected_sanity_cases) == 4
assert len(set(selected_sanity_ids)) == 4

model.eval()
for label, record in selected_sanity_cases:
    user_messages = [
        message for message in record["messages"]
        if message["role"] != "assistant"
    ]
    inputs = tokenizer.apply_chat_template(
        user_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            inputs,
            do_sample=False,
            max_new_tokens=128,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    prediction = tokenizer.decode(
        generated[0][inputs.shape[-1]:],
        skip_special_tokens=True,
    ).strip()
    print("\nCASE:", label)
    print("ID:", record["id"])
    print(
        "Query:",
        next(
            message["content"]
            for message in record["messages"]
            if message["role"] == "user"
        ),
    )
    print("Gold:", record["messages"][-1]["content"])
    print("Prediction:", prediction)

In [ ]:
# 18. ZIP adapter artifacts; optional Drive copy remains disabled
import shutil
import zipfile

ZIP_STAGING = OUTPUT_DIR / ".adapter_zip_staging"
if ZIP_STAGING.exists():
    shutil.rmtree(ZIP_STAGING)
ZIP_STAGING.mkdir(parents=True)
for filename in (
    "adapter_model.safetensors",
    "adapter_config.json",
    "training_metadata.json",
):
    shutil.copy2(OUTPUT_DIR / filename, ZIP_STAGING / filename)
if (OUTPUT_DIR / "tokenizer").is_dir():
    shutil.copytree(OUTPUT_DIR / "tokenizer", ZIP_STAGING / "tokenizer")

ZIP_PATH = REPO_ROOT / "hgfinance-risk-v1.2.zip"
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
shutil.make_archive(
    str(ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=ZIP_STAGING,
)
shutil.rmtree(ZIP_STAGING)
print("ZIP:", ZIP_PATH)
print(f"ZIP size: {ZIP_PATH.stat().st_size / 1024**2:.2f} MiB")

USE_GOOGLE_DRIVE = False
DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/HgFinance/adapters")
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ZIP_PATH, DRIVE_SAVE_DIR / ZIP_PATH.name)

## Post-training separation

This notebook trains and smoke-tests the Risk specialist adapter only. It does
not run External-50, Internal-v1, or Internal-v2 / EmployeeReasoning. Those
benchmarks remain held out for separate promotion evaluation.

The adapter is saved at root level under the external identity
hgfinance-risk-v1.2 and is not merged into the 14B base model. Production
serving remains Qwen2.5-14B-Instruct-AWQ with max_model_len=8192.